# DocBenchTR - current models on A100

Runs the pilot with newer, larger open models on a Colab **A100**. Everything except the retrieval-time model matches the published pilot and the A100 hardware control: Ollama 0.40.5, 32k context, temperature 0, the same 36 questions, the same SHA-256-pinned PDFs and the **same cached PageIndex trees** (built with `qwen2.5:7b-instruct`).

Runtime > Change runtime type > **A100 GPU**, then Runtime > Run all. Each model's results go to Google Drive (`MyDrive/docbenchtr_colab/results_a100_<model>/`) as soon as that model finishes.

In [ ]:
# Parameters: models to run, in order.
MODELS = ['qwen3.8:27b', 'gpt-oss:20b']
# Output cap per response. These models think before answering, so the cap is the full
# 32k context (the 7B pilot used 4,096).
NUM_PREDICT = 32768

In [ ]:
# 1. GPU check.
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0, 'No GPU attached. Runtime > Change runtime type > A100 GPU.'
GPU_INFO = gpu.stdout.strip()

In [ ]:
# 2. Google Drive. Stops if Drive is not authorized.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').exists(), 'Drive did not mount. Re-run this cell and approve access.'
DRIVE_ROOT = Path('/content/drive/MyDrive/docbenchtr_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive folder:', DRIVE_ROOT)

In [ ]:
# 3. Code + cached trees + pinned PDFs.
REPO_URL = 'https://github.com/umitaltinozzz/docbenchtr'
BRANCH = 'main'
REPO = Path('/content/docbenchtr')
assert 'DRIVE_ROOT' in globals(), 'Run the Drive cell first.'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('commit', COMMIT)
assert (REPO / '.pageindex' / 'docbenchtr-pilot' / 'manifest.json').exists(), 'Cached PageIndex trees missing.'

In [ ]:
# 4. Exact Python environment from uv.lock.
assert 'REPO' in globals(), 'Run the checkout cell first.'
!pip -q install uv
!cd /content/docbenchtr && uv sync --frozen --python 3.12 2>&1 | tail -n 3
PY = str(REPO / '.venv' / 'bin' / 'python')
!{PY} -c "import torch, pageindex; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"


In [ ]:
# 5. Ollama 0.40.5 with a 32k context. Models are cached in Drive.
import os, time, json, urllib.request
assert 'PY' in globals(), 'Run the environment cell first.'
OLLAMA_VERSION = '0.40.5'
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > /tmp/ollama-install.log 2>&1; tail -n 2 /tmp/ollama-install.log
(DRIVE_ROOT / 'ollama_models').mkdir(parents=True, exist_ok=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='32768', OLLAMA_HOST='127.0.0.1:11434',
           OLLAMA_MODELS=str(DRIVE_ROOT / 'ollama_models'), OLLAMA_MAX_LOADED_MODELS='1')
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
    print('Ollama already running')
except Exception:
    server = subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama-server.log', 'a'),
                              stderr=subprocess.STDOUT, env=env)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start; see /tmp/ollama-server.log')
os.environ.update({k: env[k] for k in ('OLLAMA_CONTEXT_LENGTH', 'OLLAMA_HOST', 'OLLAMA_MODELS')})
!ollama --version
OLLAMA_READY = True

In [ ]:
# 6. Run every model. Each model: pull, apply the pilot's Modelfile settings, run, re-run failed
#    Tree Search rows, copy results to Drive. Trees must be REUSED, never rebuilt.
import shutil
from datetime import datetime, timezone
assert globals().get('OLLAMA_READY'), 'Run the Ollama cell first.'
done = []
for model in MODELS:
    slug = model.replace(':', '-').replace('/', '-')
    target = DRIVE_ROOT / f'results_a100_{slug}'
    if (target / 'summary.json').exists():
        print(f'== {model}: already on Drive, skipping'); done.append(model); continue
    print(f'== {model}: pull', flush=True)
    subprocess.run(['ollama', 'pull', model], check=True)
    Path('/tmp/Modelfile').write_text(f'FROM {model}\nPARAMETER num_ctx 32768\nPARAMETER num_predict {NUM_PREDICT}\n')
    subprocess.run(['ollama', 'create', model, '-f', '/tmp/Modelfile'], check=True)
    shutil.rmtree(REPO / 'results', ignore_errors=True)
    run_env = dict(os.environ, DOCBENCHTR_OLLAMA_MODEL=model, DOCBENCHTR_NUM_PREDICT=str(NUM_PREDICT))
    log_path = DRIVE_ROOT / f'run_a100_{slug}.log'
    reused = 0
    proc = subprocess.Popen([PY, '-u', '-m', 'docbenchtr.run_pilot'], cwd=REPO, env=run_env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(log_path, 'w') as log:
        for line in proc.stdout:
            log.write(line); log.flush()
            if 'Batches' in line:
                continue
            print(line, end='')
            if 'Reused verified PageIndex tree' in line:
                reused += 1
    proc.wait()
    assert proc.returncode == 0, f'{model} failed; see {log_path}'
    assert reused >= 3, f'{model}: cached trees were not reused; not comparable.'
    rows = json.loads((REPO / 'results' / 'per_query.json').read_text())
    if any(r['method'] == 'PageIndex LLM Tree Search' and r['status'] == 'failed' for r in rows):
        subprocess.run([PY, '-u', '-m', 'docbenchtr.rerun_failed', 'PageIndex LLM Tree Search'],
                       cwd=REPO, env=run_env, check=True)
    record = {
        'model': model, 'gpu': GPU_INFO, 'commit': COMMIT, 'ollama_version': OLLAMA_VERSION,
        'modelfile': subprocess.run(['ollama', 'show', model, '--modelfile'], capture_output=True, text=True).stdout,
        'ollama_list': subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout,
        'context_length': 32768, 'num_predict': NUM_PREDICT,
        'finished_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }
    (REPO / 'results' / 'run_environment.json').write_text(json.dumps(record, indent=2))
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(REPO / 'results', target)
    subprocess.run(['ollama', 'stop', model])
    done.append(model)
    print(f'== {model}: saved to {target}', flush=True)
print('finished:', done)

In [ ]:
# 7. Zip all model results and download.
import shutil
bundle = Path('/content/docbenchtr_results_a100_models')
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir()
for model in MODELS:
    slug = model.replace(':', '-').replace('/', '-')
    src = DRIVE_ROOT / f'results_a100_{slug}'
    if src.exists():
        shutil.copytree(src, bundle / slug)
shutil.copy('/tmp/ollama-server.log', bundle / 'ollama-server.log')
archive = shutil.make_archive(str(bundle), 'zip', bundle)
from google.colab import files
files.download(archive)